# Rapport Meteo - 3 Grandes Villes du Canada

---

| Champ | Valeur |
|-------|--------|
| **Etudiant** | Babatunde Adissa Fadolle Arouna - 300151970 |
| **Hiver** | 2026 |
| **Programme** | Techniques des systemes informatiques |
| **Cours** | INF 1102-201 Programmation de systemes |
| **Professeur** | Brice Robert |

---

## Objectif

Ce notebook analyse et visualise les donnees meteo de
**Toronto**, **Montreal** et **Vancouver** recuperees via
l API OpenWeatherMap.

Il genere 4 graphiques :
- 2 diagrammes en bandes (temperatures / humidite-vent)
- 2 diagrammes circulaires (humidite / nuages)

## Pipeline d execution

| Etape | Outil | Terminal | Action |
|-------|-------|----------|--------|
| 1 | PowerShell | Windows | analyse.ps1 appelle l API et sauvegarde JSON |
| 2 | Bash | VM SSH | analyse.sh verifie et lance Python |
| 3 | Python | VM SSH | analyse.py genere graphiques + rapport.txt |
| 4 | Jupyter | VM SSH | Ce notebook visualise les resultats |

## 1. Importation des bibliotheques

On importe toutes les bibliotheques necessaires pour
la lecture JSON, les calculs et la generation des graphiques.

In [ ]:
# Bibliotheques standard Python
import json
import os
import datetime

# Bibliotheque de calcul numerique
import numpy as np

# Bibliotheque de visualisation
import matplotlib
import matplotlib.pyplot as plt

print('Bibliotheques chargees avec succes.')
print(f'matplotlib version : {matplotlib.__version__}')
print(f'numpy version      : {np.__version__}')

## 2. Chargement des donnees JSON

Le fichier `data/meteo_raw.json` contient les donnees brutes
de l API OpenWeatherMap pour les 3 villes canadiennes.
Il est genere automatiquement par `scripts/analyse.ps1`.

In [ ]:
# Chargement du fichier JSON
json_path = os.path.join('data', 'meteo_raw.json')
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

villes_data = data['villes']

print(f'Fichier charge : {json_path}')
print(f'Nombre de villes : {len(villes_data)}')
print('')
print('Apercu des donnees :')
for v in villes_data:
    print(f"  {v['name']:<12} : {v['main']['temp']} C | {v['weather'][0]['description']}")

## 3. Extraction et affichage des donnees

On extrait toutes les donnees utiles du JSON pour
faciliter la creation des graphiques et du rapport.

In [ ]:
# Extraction des donnees pour chaque ville
villes         = [v['name'] for v in villes_data]
temps          = [v['main']['temp'] for v in villes_data]
temp_min       = [v['main']['temp_min'] for v in villes_data]
temp_max       = [v['main']['temp_max'] for v in villes_data]
temp_ressentie = [v['main']['feels_like'] for v in villes_data]
humidites      = [v['main']['humidity'] for v in villes_data]
pressions      = [v['main']['pressure'] for v in villes_data]
nuages         = [v['clouds']['all'] for v in villes_data]
vents          = [round(v['wind']['speed'] * 3.6, 1) for v in villes_data]
conditions     = [v['weather'][0]['description'].capitalize() for v in villes_data]
visibilite     = [v.get('visibility', 0) / 1000 for v in villes_data]
lever          = [datetime.datetime.fromtimestamp(v['sys']['sunrise']).strftime('%H:%M') for v in villes_data]
coucher        = [datetime.datetime.fromtimestamp(v['sys']['sunset']).strftime('%H:%M') for v in villes_data]
couleurs_villes= ['#FF6B6B', '#4FC3F7', '#81C784']

# Affichage complet des donnees extraites
print(f'{"Ville":<12} {"Temp":>6} {"Ressenti":>9} {"Min":>5} {"Max":>5} {"Hum%":>5} {"Nua%":>5} {"Vent":>7} {"Condition"}')
print('-' * 75)
for i, v in enumerate(villes):
    print(f'{v:<12} {temps[i]:>6.1f} {temp_ressentie[i]:>9.1f} {temp_min[i]:>5.1f} {temp_max[i]:>5.1f} '
          f'{humidites[i]:>5} {nuages[i]:>5} {vents[i]:>7} {conditions[i]}')

## 4. Diagramme en bandes - Comparaison des temperatures

Ce graphique compare les 4 valeurs de temperature
(actuelle, ressentie, minimale, maximale) pour chaque ville.

**Commande de generation :**
```bash
python3 scripts/analyse.py data/meteo_raw.json output/rapport.txt
```

In [ ]:
# Diagramme en bandes groupees - Temperatures
fig, ax = plt.subplots(figsize=(11, 6))
x     = np.arange(len(villes))
width = 0.2

# 4 series de barres cote a cote
b1 = ax.bar(x - width*1.5, temps,          width, label='Actuelle',  color='#FF6B6B', edgecolor='black')
b2 = ax.bar(x - width*0.5, temp_ressentie, width, label='Ressentie', color='#FFA07A', edgecolor='black')
b3 = ax.bar(x + width*0.5, temp_min,       width, label='Minimale',  color='#87CEEB', edgecolor='black')
b4 = ax.bar(x + width*1.5, temp_max,       width, label='Maximale',  color='#FF4500', edgecolor='black')

# Affichage des valeurs sur les barres
for bars in [b1, b2, b3, b4]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Comparaison des temperatures - Toronto, Montreal, Vancouver',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xticks(x)
ax.set_xticklabels(villes, fontsize=12)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.legend()
ax.set_ylim(0, max(temp_max)+5)
plt.tight_layout()
plt.savefig(os.path.join('output', 'graph_barres_temperatures.png'), dpi=150)
plt.show()
print('Graphique sauvegarde : output/graph_barres_temperatures.png')

**Analyse et commentaires :**

- **Vancouver** est la ville la plus chaude avec 13.5 C actuels
- **Montreal** se situe au milieu avec 11.2 C
- **Toronto** est la plus fraiche avec 9.4 C
- L amplitude min/max est similaire pour les 3 villes (environ 4 C)
- La temperature ressentie est inferieure a l actuelle dans toutes les villes a cause du vent

**Capture d ecran :** Faites une capture de ce graphique et sauvegardez-la dans `images/capture_graphique_barres.png`

## 5. Diagramme en bandes - Humidite, Nuages et Vent

Ce graphique compare trois indicateurs atmospheriques
cles pour chaque ville : humidite, couverture nuageuse et vent.

In [ ]:
# Diagramme en bandes groupees - Humidite, Nuages, Vent
fig, ax = plt.subplots(figsize=(11, 6))
x2     = np.arange(len(villes))
width2 = 0.25

b5 = ax.bar(x2 - width2, humidites, width2, label='Humidite (%)', color='#4FC3F7', edgecolor='black')
b6 = ax.bar(x2,          nuages,    width2, label='Nuages (%)',   color='#B0BEC5', edgecolor='black')
b7 = ax.bar(x2 + width2, vents,     width2, label='Vent (km/h)', color='#81C784', edgecolor='black')

for bars in [b5, b6, b7]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Comparaison Humidite, Nuages et Vent - 3 villes canadiennes',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Valeur')
ax.set_xticks(x2)
ax.set_xticklabels(villes, fontsize=12)
ax.legend()
ax.set_ylim(0, max(humidites+nuages+vents)*1.25)
plt.tight_layout()
plt.savefig(os.path.join('output', 'graph_barres_humidite_vent.png'), dpi=150)
plt.show()
print('Graphique sauvegarde : output/graph_barres_humidite_vent.png')

**Analyse et commentaires :**

- **Vancouver** est la ville la plus humide (88%) et la plus nuageuse (90%),
  typique de son climat oceanique pacifique avec pluie moderee
- **Montreal** est la plus seche (58%) avec un ciel presque degage (10%),
  conditions ideales pour les activites exterieures
- **Toronto** est en position intermediaire avec 72% d humidite et 75% de nuages
- Le vent le plus fort est a Vancouver (20.9 km/h) venant de l ouest

**Capture d ecran :** Sauvegardez dans `images/capture_graphique_barres.png`

## 6. Diagramme circulaire - Repartition de l humidite

Ce graphique montre la proportion d humidite de chaque
ville par rapport au total des 3 villes combinees.

In [ ]:
# Diagramme circulaire - Humidite des 3 villes
fig, ax = plt.subplots(figsize=(7, 7))
labels3 = [f'{v}\n{h}%' for v, h in zip(villes, humidites)]

wedges, texts, auto = ax.pie(
    humidites,
    labels=labels3,
    colors=couleurs_villes,
    explode=(0.05, 0.05, 0.05),
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12}
)
for a in auto:
    a.set_fontweight('bold')

ax.set_title('Repartition de l humidite\nToronto vs Montreal vs Vancouver',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('output', 'graph_circulaire_humidite.png'), dpi=150)
plt.show()
print('Graphique sauvegarde : output/graph_circulaire_humidite.png')

**Analyse et commentaires :**

- **Vancouver** domine avec 40.4% de l humidite totale combinee
- **Toronto** represente 33.2% de l humidite totale
- **Montreal** est la plus seche avec seulement 26.7%
- Ce graphique confirme que le climat pacifique de Vancouver
  est significativement plus humide que les villes de l est

**Capture d ecran :** Sauvegardez dans `images/capture_graphique_circulaire.png`

## 7. Diagramme circulaire - Couverture nuageuse

Ce graphique montre la repartition de la couverture
nuageuse entre les 3 villes.

In [ ]:
# Diagramme circulaire - Couverture nuageuse
fig, ax = plt.subplots(figsize=(7, 7))
labels4  = [f'{v}\n{n}%' for v, n in zip(villes, nuages)]
nuages_safe = [max(n, 1) for n in nuages]  # Eviter valeur 0

wedges4, texts4, auto4 = ax.pie(
    nuages_safe,
    labels=labels4,
    colors=couleurs_villes,
    explode=(0.05, 0.05, 0.05),
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12}
)
for a in auto4:
    a.set_fontweight('bold')

ax.set_title('Repartition de la couverture nuageuse\nToronto vs Montreal vs Vancouver',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('output', 'graph_circulaire_nuages.png'), dpi=150)
plt.show()
print('Graphique sauvegarde : output/graph_circulaire_nuages.png')

**Analyse et commentaires :**

- **Vancouver** concentre 51.7% de la couverture nuageuse totale
- **Toronto** represente 43.5% avec un ciel moderement nuageux
- **Montreal** n a que 5.8% de couverture nuageuse (ciel degage)
- Cette difference marquee entre Vancouver et Montreal illustre
  parfaitement la difference de climat entre la cote Pacifique
  et le Quebec continental

**Capture d ecran :** Sauvegardez dans `images/capture_graphique_circulaire.png`

## 8. Lecture du rapport texte genere automatiquement

Le fichier `output/rapport.txt` est genere automatiquement
par `scripts/analyse.py` lors de l execution.

In [ ]:
# Lecture et affichage du rapport texte
rapport_path = os.path.join('output', 'rapport.txt')
if os.path.exists(rapport_path):
    with open(rapport_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('ATTENTION : rapport.txt non trouve.')
    print('Lancez d abord : bash scripts/analyse.sh')

**Capture d ecran :** Faites une capture de ce resultat et sauvegardez-la dans `images/capture_rapport.png`

## 9. Synthese et conclusion

### Tableau comparatif des 3 villes

| Ville | Temperature | Ressentie | Humidite | Nuages | Vent | Condition |
|-------|-------------|-----------|----------|--------|------|----------|
| Toronto | 9.4 C | 6.8 C | 72% | 75% | 14.8 km/h | Nuageux |
| Montreal | 11.2 C | 9.0 C | 58% | 10% | 11.5 km/h | Ciel degage |
| Vancouver | 13.5 C | 11.8 C | 88% | 90% | 20.9 km/h | Pluie moderee |

### Conclusions

- **Vancouver** est la ville la plus chaude mais aussi la plus humide
  et nuageuse, avec de la pluie moderee due au climat oceanique
- **Montreal** beneficie du meilleur temps ce jour avec un ciel
  presque completement degage et une humidite moderee
- **Toronto** se situe entre les deux avec des conditions
  moderement nuageuses et une humidite acceptable

### Technologies utilisees

| Outil | Terminal | Role |
|-------|----------|------|
| PowerShell | Windows | Appel API REST OpenWeatherMap |
| Bash | VM Ubuntu SSH | Orchestration du pipeline |
| Python 3 | VM Ubuntu SSH | Analyse JSON + 4 graphiques |
| Jupyter | VM Ubuntu SSH | Rapport interactif |

### Points cles appris

- Connexion SSH a une VM Ubuntu depuis PowerShell Windows
- Appel d une API REST publique avec Invoke-RestMethod
- Lecture et traitement de donnees JSON en Python
- Generation de diagrammes en bandes et circulaires avec matplotlib
- Graphiques groupes avec numpy pour comparer plusieurs series
- Generation automatique d un rapport texte
- Organisation d un projet multi-langage complet

---

*INF 1102-201 Programmation de systemes - Hiver 2026*

*Babatunde Adissa Fadolle Arouna | 300151970 | Professeur : Brice Robert*